In [4]:
import torch     
import torch.nn as nn
import torch.nn.functional as F

**DeepSeek Style MoE**

In [5]:
d_model = 512

# MoE Specific Params for Fine-Grained Expert Segmentation
n_routed_experts: int = 16
top_k: int = 2
expert_hidden_dim: int = 512 # (n_embd * 4) / 4
dropout = 0.0

# Shared Expert Isolation
n_shared_experts: int = 2
shared_expert_hidden_dim: int = 1024 # Larger for generalist role

In [3]:
class Expert(nn.Module):
    """ A standard MLP expert for the routed experts. Uses GELU activation. """
    def __init__(self, n_embd: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, hidden_dim, bias=False),
            nn.GELU(),
            nn.Linear(hidden_dim, n_embd, bias=False),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class SharedExpert(nn.Module):
    """ Shared experts. """
    def __init__(self, n_embd: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.w1 = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w3 = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(self.dropout(F.silu(self.w1(x)) * self.w3(x)))

In [ ]:
class DeepSeekMoELayer(nn.Module):
  def __init__(self, d_model, n_routed_experts, n_shared_experts, top_k, dropout=0.0):
    super().__init__()
    self.n_routed_experts = n_routed_experts
    self.n_shared_experts = n_shared_experts
    self.top_k = top_k

    # shared experts
    self.shared_experts = nn.ModuleList(
        [ SharedExpert(d_model, shared_expert_hidden_dim, dropout=dropout) for _ in range(self.n_shared_experts) ]
    )

    # routed experts
    self.routed_experts = nn.ModuleList(
        [ Expert(d_model, expert_hidden_dim, dropout=dropout) for _ in range(self.n_routed_experts) ]
    )

    # Router Matrix: A simple linear layer that outputs logits
    self.router = nn.Linear(d_model, self.n_routed_experts, bias=False)

    # Non-trainable buffer for the auxiliary-loss-free bias term
    self.register_buffer("router_bias", torch.zeros(self.n_routed_experts))

    # Bias update parameters (as per reference)
    self.bias_update_gamma = 1e-3

    # Bias update parameters (as per reference)
    self.bias_update_gamma = 1e-3 # Corresponds to gamma_initial
    self.clamp_abs = 2.0 # Optional clamping for stability
    self.tol_frac = 0.0 # Tolerance fraction, 0.0 means we always update if not perfectly balanced

    # For interpretability
    self.router_indices = None

  def _affinity(self, x_flat: torch.Tensor):
    """
    FIX 1: Compute s = sigmoid(linear(x)) in fp32.
    The router logits are computed in the model's dtype (e.g., bfloat16) for performance,
    but the sigmoid and resulting affinity scores 's' are kept in float32 for numerical stability.
    """
    router_logits = self.router(x_flat)
    return torch.sigmoid(router_logits.float()) # Return fp32 affinity scores
  
  def forward(self, x, analysis_mode=False):
    B, T, C = x.shape
    x_flat = x.view(-1, C)   # [N, C]  # N = no. of tokens = B*T

    # --- 1. Shared Expert Path (Dense) ---
    shared_out_flat = torch.zeros_like(x_flat)
    for expert in self.shared_experts:
      shared_out_flat += expert(x_flat)
    
    # --- 2. Routed Expert Path (Sparse) ---
    # 2a. Calculate affinity scores 's' and select Top-K using 's + b'
    s = self._affinity(x_flat) # [N, E] in fp32
    sel_scores = s + self.router_bias.to(s.device) # Add bias for selection
    topk_indices = torch.topk(sel_scores, self.top_k, dim=-1).indices # [N, K]

    # 2b. Calculate gates from the original
    # 2b. Calculate gates from the original, unbiased 's'
    # FIX 2: Gates = normalize(s.gather(TopK)). Bias is NEVER included in gates.
    s_sel = s.gather(dim=1, index=topk_indices) # [N, K]
    denom = s_sel.sum(dim=1, keepdim=True)
    # Fallback to 1/K if sum is tiny to prevent division by zero
    gates = torch.where(
        denom > 1e-9,
        s_sel / (denom + 1e-9),
        torch.full_like(s_sel, 1.0 / self.top_k)
        ).to(x.dtype) # [N, K]

    # For interpretability
    if analysis_mode:
      self.router_indices = topk_indices.detach().cpu()

    # 2c. Dispatch tokens and combine outputs
    # FIX 5: Correct, efficient dispatch logic.
    routed_out_flat = torch.zeros_like(x_flat)
    for i in range(self.n_routed_experts):
      # Create a mask for tokens routed to expert 'i'
      mask = (topk_indices == i)
      row_idx, which_k = mask.nonzero(as_tuple=True)

      if row_idx.numel() == 0:
        continue

      # Select the input tokens for this expert
      expert_in = x_flat.index_select(0, row_idx)
      # Run the expert
      expert_out = self.routed_experts[i](expert_in)
      # Select the gating weights for these specific tokens and expert
      gate_values = gates[row_idx, which_k].unsqueeze(1)
      # Weight the expert output by its gate
      weighted_expert_out = expert_out * gate_values
      # Add the result back to the final output tensor at the correct positions
      routed_out_flat.index_add_(0, row_idx, weighted_expert_out)

    # --- 3. Final Combination ---
    # FIX 3: Add residual 'x' in the final output.
    final_output = x + shared_out_flat.view(B, T, C) + routed_out_flat.view(B, T, C)
    return final_output 


    

In [7]:
class Block(nn.Module):
    """ A single Transformer block combining Multi-Head Attention and our DeepSeekMoE layer. """
    def __init__(self, ):
        super().__init__()
        self.ln_1 = nn.LayerNorm()
        self.attn = MultiHeadAttention()
        self.ln_2 = nn.LayerNorm()
        self.moe = DeepSeekMoELayer(d_model, n_routed_experts, n_shared_experts, top_k, dropout=0.0)

    # <--- MODIFIED: This function now returns the MoE input as well
    def forward(self, x: torch.Tensor, analysis_mode=False):
        x = x + self.attn(self.ln_1(x))
        # Capture the input to the MoE layer before it's processed
        moe_input = self.ln_2(x)
        # The MoE layer's output is the FFN part, which we add to the residual path
        x = x + self.moe(moe_input, analysis_mode=analysis_mode)
        # Return both the final output and the captured input for the bias update
        return x, moe_input